In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del archivo "language_role.json"

###Paso 1 - Leer el archivo JSON usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
language_role_schema = StructType([
    StructField("languageRole", StringType(), True),
    StructField("roleId", IntegerType(), True)
])

In [0]:
language_role_df = spark.read\
    .option("multiline", True)\
    .schema(language_role_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/language_role.json")

### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
language_role_final_df = add_ingestion_date(language_role_df)\
    .withColumnsRenamed({"languageRole": "language_role",
                         "roleId": "role_id"})\
    .withColumn("enviroments", lit("Production"))\
    .withColumn("file_date", lit(v_file_date))

### Paso 3 - Escribir la salida en un formato "Parquet" PartitionBy

In [0]:
#language_role_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/language_roles")

In [0]:
language_role_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.language_roles")

In [0]:
%sql
SELECT * FROM movie_silver.language_roles

language_role,role_id,ingestion_date,enviroments,file_date
Original,1,2026-09-08T01:16:06.031402Z,Production,2024-12-16
Spoken,2,2026-09-08T01:16:06.031402Z,Production,2024-12-16


In [0]:
dbutils.notebook.exit("Exitoso")